# **FAIML: Reinforcement Learning project**

## **Part 2 - Task 6: Domain Randomization**

Train a UDR and ADR agent on the source environment with the same RL algorithm previously used. Later test the policy obtained on both the source and target environments.

*Guiding Questions*
*   Is UDR able to overcome the shift of mass and lead to more robust policies w.r.t. the naive “source→target” baseline in task 5?
* Can you think of limitations or downsides of UDR? What about ADR? Are there limitations/assumptions here?

*Hints:* \
Remember not to act like you already know the target mass! Choosing a very narrow distribution for UDR around the target mass is basically cheating! We want a robust policy, not another mass-specific one!


### **Project setup**
Let's start by setting up the environment for the project, we will be cloning the github repository and importing the necessary libraries for the tasks we will handle

In [ ]:
# cloning the given github reporistory
!git clone https://github.com/lambdavi/FAIML-RL-26.git

%cd FAIML-RL-26/part2/panda-gym
!pip install -e .

%cd ..

Cloning into 'FAIML-RL-26'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 124 (delta 4), reused 2 (delta 2), pack-reused 109 (from 1)
Receiving objects: 100% (124/124), 8.34 MiB | 23.35 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/FAIML-RL-26/part2/FAIML-RL-26/part2/panda-gym
Obtaining file:///content/FAIML-RL-26/part2/FAIML-RL-26/part2/panda-gym
  Preparing metadata (setup.py) ... done
  Attempting uninstall: panda_gym
    Found existing installation: panda_gym 3.0.8
    Uninstalling panda_gym-3.0.8:
      Successfully uninstalled panda_gym-3.0.8
  Running setup.py develop for panda_gym


/content/FAIML-RL-26/part2/FAIML-RL-26/part2


In [ ]:
!pip install stable-baselines3

### **Domain Randomization Wrapper**

The following code for the wrapper used to implement UDR and ADR is based on the code provided in the github repository. Both algorithms share a common class, an attribute decides which approach to implement.

In [ ]:
import gymnasium as gym
import numpy as np

# a useful buffer library we can use to implement buffers,
# it is more efficient than a list and a numpy array for adding values
from collections import deque

class RandomizationWrapper(gym.Wrapper):
    """
    Wrapper that applies randomization to the environment.
    """
    def __init__(
        self,
        env,
        mass_range=(0.1, 20),
        mode="UDR",
        # ADR parameters
        delta=0.1, # the delta we use to expand or shrink the domain of uniform distribution from which we draw the env's parameters (in this project only the mass)
        t_low=0.5, # lower threshold, if perfs are lower than this we shrink the distribution's domain
        t_high=0.8, # upper threshold, same logice for enlarging domain
        buffer_size=15, #  max number of performances recorded before doing the average performance
        boundary_prob=0.5, # probability of sampling withinng the bounday or the boundaries themselves
        # avoid having negative masses by implementing an absolute lower limit
        mass_min_limit=0.1,
        # eventhough unlikely, avoid having to push around a ver heavy mass
        mass_max_limit=30.0,
    ):
        super().__init__(env)

        self.mode = mode
        self.mass_range = mass_range

        # interval boundaries
        self.mass_min, self.mass_max = mass_range
        self.mass_min_limit = mass_min_limit
        self.mass_max_limit = mass_max_limit

        self.last_sample_type = None

        # ADR parameters
        self.delta = delta
        self.t_low = t_low
        self.t_high = t_high
        self.buffer_size = buffer_size
        self.boundary_prob = boundary_prob
        self._episode_reward = 0.0

        # performance buffers for upper and lower bounds
        self.buffer_low = deque(maxlen=buffer_size)
        self.buffer_high = deque(maxlen=buffer_size)

    # -----------------------
    # Mass Sampling
    # -----------------------

    def _sample_mass(self):

        # for UDR we simply sample mass from unifom distribution
        if self.mode == "UDR":
            self.last_sample_type = None
            return np.random.uniform(self.mass_min, self.mass_max)


        elif self.mode == "ADR":
          # we run a samping from uniform distribution to see if we use algo 1 or 2
          if np.random.uniform(0, 1) < self.boundary_prob:
              # Algorithm 1 : boundary sampling
              if np.random.uniform(0, 1) < 0.5:
                  self.last_sample_type = "low"
                  return self.mass_min
              else:
                  self.last_sample_type = "high"
                  return self.mass_max
          else:
              # Algorithm 2 : training data generation, sampling data within boundaries
              self.last_sample_type = "interior"
              return np.random.uniform(self.mass_min, self.mass_max)

        else:
            raise NotImplementedError(f"Sampling strategy '{self.mode}' is not implemented yet.")

    def _update_bounds(self, performance):
        """updating the boundaries of the mass depending on the results in buffer"""

        if self.last_sample_type == "low":
          # at the end of each episode we append result to lower or upper buffer
            self.buffer_low.append(performance)
            # when buffer is full
            if len(self.buffer_low) >= self.buffer_size:
                mean_perf = np.mean(self.buffer_low) # compute mean performance
                self.buffer_low.clear() # clear the buffer
                if mean_perf >= self.t_high:
                    # if perfs are good we enlarge the lower boundary of the mass
                    self.mass_min = max(self.mass_min_limit, self.mass_min - self.delta)
                elif mean_perf <= self.t_low:
                    # otherwise shrink them
                    self.mass_min = min(self.mass_min + self.delta, self.mass_max)
                print(f"[ADR] Updated mass_min={self.mass_min:.2f}")

        # same logic for the upper boundary of the mass
        elif self.last_sample_type == "high":
            self.buffer_high.append(performance)
            if len(self.buffer_high) >= self.buffer_size:
                mean_perf = np.mean(self.buffer_high)
                self.buffer_high.clear()
                if mean_perf >= self.t_high:
                    self.mass_max = min(self.mass_max_limit, self.mass_max + self.delta)
                elif mean_perf <= self.t_low:
                    self.mass_max = max(self.mass_max - self.delta, self.mass_min)
                print(f"[ADR] Updated mass_max={self.mass_max:.2f}")

    def step(self, action):
        """at the end of every episode we update boundaries"""
        obs, reward, terminated, truncated, info = self.env.step(action)

        # accumulate the rewards
        self._episode_reward += float(reward)

        if self.mode == "ADR" and (terminated or truncated):
            if info.get("is_success", False):
              performance = 1.0
            else:
                # for dense reward is equal to -(dist) and the maximum dist is 8
                # considering the max number of steps per episode and the biggest
                # distance possible being 0.141
                performance = float(np.clip((self._episode_reward + 8) / 8, 0.0, 1.0))
            self._update_bounds(performance)

        return obs, reward, terminated, truncated, info

    # -----------------------
    # Reset
    # -----------------------

    def reset(self, **kwargs):
        """method to reset the environment"""
        new_mass = self._sample_mass()
        self._episode_reward = 0.0

        sim = self.env.unwrapped.task.sim
        object_body_id = sim._bodies_idx["object"]

        sim.physics_client.changeDynamics(
            bodyUniqueId=object_body_id,
            linkIndex=-1,
            mass=float(new_mass),
        )

        print(
            f"[{self.mode}] mass={new_mass:.2f} "
            f"range=[{self.mass_min:.2f},{self.mass_max:.2f}] "
            f"type={self.last_sample_type}"
        )

        return super().reset(**kwargs)

### **Training a UDR agent with SAC model in source environment**

In [ ]:
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from google.colab import drive
np.random.seed(42)

drive.mount('/content/drive')

# initializing the source environment wrapped with UDR
env = gym.make(
    "PandaPush-v3",
    render_mode="rgb_array",
    reward_type="dense",
    type="source"
)
env = RandomizationWrapper(env, mode="UDR")

# creating a CallBack instance to save progress
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path="/content/drive/MyDrive/RL_models/checkpoints_udr/",
)

# creating an instance of the SAC model and do training on selected env
SAC_udr = SAC(
    "MultiInputPolicy",
    env,
    learning_rate=1e-4,
    batch_size=256,
    verbose=1,
    tensorboard_log="./tb_logs/",
    seed = 42
)
SAC_udr.learn(total_timesteps=1_000_000, callback=checkpoint_callback)

SAC_udr.save("/content/drive/MyDrive/RL_models/sac_udr")

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
|    learning_rate   | 0.0001   |
|    n_updates       | 79977    |
---------------------------------
[UDR] mass=17.63 range=[0.10,20.00] type=None
[UDR] mass=7.73 range=[0.10,20.00] type=None
[UDR] mass=18.48 range=[0.10,20.00] type=None
[UDR] mass=5.35 range=[0.10,20.00] type=None
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 42.5     |
|    ep_rew_mean     | -3.96    |
|    success_rate    | 0.16     |
| time/              |          |
|    episodes        | 1948     |
|    fps             | 25       |
|    time_elapsed    | 3174     |
|    total_timesteps | 80229    |
| train/             |          |
|    actor_loss      | 6.27     |
|    critic_loss     | 0.026    |
|    ent_coef        | 0.00211  |
|    ent_coef_loss   | -1.52    |
|    learning_rate   | 0.0001   |
|    n_updates       | 80128    |
---------------------------------
[UDR] mass=8.10 range=[0.1

In [ ]:
# training got interrupted so we continue from last checkpoint:
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from google.colab import drive
np.random.seed(42)

drive.mount('/content/drive')

# initializing the source environment wrapped with UDR
env = gym.make(
    "PandaPush-v3",
    render_mode="rgb_array",
    reward_type="dense",
    type="source"
)
env = RandomizationWrapper(env, mode="UDR")

# creating a CallBack instance to save progress
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path="/content/drive/MyDrive/RL_models/checkpoints_udr/",
)
# loading last checkpoint checkpoint
SAC_udr = SAC.load(
    "/content/drive/MyDrive/RL_models/checkpoints_udr/rl_model_910000_steps",
    env=env
)

SAC_udr.learn(
    total_timesteps=90_000,  # remaining steps to each a million
    callback=checkpoint_callback,
    reset_num_timesteps=False  # to continue the counter from where we were
)

SAC_udr.save("/content/drive/MyDrive/RL_models/sac_udr")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Mounted at /content/drive
Created object with mass: 1.0
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
[UDR] mass=1.59 range=[0.10,20.00] type=None
Logging to ./tb_logs/SAC_0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
|    critic_loss     | 0.0565   |
|    ent_coef        | 0.00535  |
|    ent_coef_loss   | -1.03    |
|    learning_rate   | 0.0001   |
|    n_updates       | 968632   |
---------------------------------
[UDR] mass=18.50 range=[0.10,20.00] type=None
[UDR] mass=19.74 range=[0.10,20.00] type=None
[UDR] mass=10.60 range=[0.10,20.00] type=None
[UDR] mass=13.12 range=[0.10,20.00] type=None
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 33.7     |
|    ep_rew_mean     | -2.97    |
|    success_rate    | 0.37     |
| time/              |          |
|    episodes        | 24624    |
|    fps             | 43       |
|    time_elapsed    | 1344     |
|    total_timesteps | 968846   |
| train/             |          |
|    actor_loss      | 6.91     |
|    critic_loss     | 0.0307   |
|    ent_coef        | 0.00536  |
|    ent_coef_loss   | 0.488    |
|    learning_rate   | 0

Obtained results are unstatisfying, we will try a narrower mass distribution

### **Second attempt of UDR with a narrower mass distribution**

In [ ]:
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from google.colab import drive

drive.mount('/content/drive')

# initializing the source environment wrapped with UDR
env = gym.make(
    "PandaPush-v3",
    render_mode="rgb_array",
    reward_type="dense",
    type="source",
)
env = RandomizationWrapper(env, mode="UDR", mass_range=(0.1,12))

# creating a CallBack instance to save progress
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path="/content/drive/MyDrive/RL_models/checkpoints_udr_narrow/",
)

# creating an instance of the SAC model and do training on selected env
SAC_udr_narrow = SAC(
    "MultiInputPolicy",
    env,
    learning_rate=1e-4,
    batch_size=256,
    verbose=1,
    tensorboard_log="./tb_logs/",
    seed = 42
)
SAC_udr_narrow.learn(total_timesteps=1_000_000, callback=checkpoint_callback)

SAC_udr_narrow.save("/content/drive/MyDrive/RL_models/sac_udr_narrow")

In [ ]:
# training got interrupted so we continue from last checkpoint:
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from google.colab import drive
import numpy as np
import random
import torch

#connecting to drive
drive.mount('/content/drive')

# setting seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# initializing the source environment wrapped with UDR
env = gym.make(
    "PandaPush-v3",
    render_mode="rgb_array",
    reward_type="dense",
    type="source"
)
env = RandomizationWrapper(env, mode="UDR", mass_range=(0.1,12))

# creating a CallBack instance to save progress
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path="/content/drive/MyDrive/RL_models/checkpoints_udr_narrow/",
)
# loading last checkpoint checkpoint
SAC_udr_narrow = SAC.load(
    "/content/drive/MyDrive/RL_models/checkpoints_udr_narrow/rl_model_750000_steps",
    env=env,
    seed = 42
)

SAC_udr_narrow.learn(
    total_timesteps=250_000,  # remaining steps to each a million
    callback=checkpoint_callback,
    reset_num_timesteps=False  # to continue the counter from where we were
)

SAC_udr_narrow.save("/content/drive/MyDrive/RL_models/sac_udr_narrow")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Created object with mass: 1.0


NameError: name 'RandomizationWrapper' is not defined

### **Training an ADR agent with a SAC model in source environment**
The following model was trained for the first time with ADR using the same wide starting mass interval. It has collapsed leading to a degenerate mass interval which is why we didn't take this into account in our experiment

In [ ]:
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from google.colab import drive

drive.mount('/content/drive')

# initializing the source environment wrapped with ADR
env = gym.make(
    "PandaPush-v3",
    render_mode="rgb_array",
    reward_type="dense",
    type="source"
)
env = RandomizationWrapper(env, mode="ADR")

# creating a CallBack instance to save progress
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path="/content/drive/MyDrive/RL_models/checkpoints_adr/",
)

# creating an instance of the SAC model and do training on selected env
SAC_adr = SAC(
    "MultiInputPolicy",
    env,
    learning_rate=1e-4,
    batch_size=256,
    verbose=1,
    tensorboard_log="./tb_logs/"
    seed = 42
)
SAC_adr.learn(total_timesteps=1_000_000, callback=checkpoint_callback)

SAC_adr.save("/content/drive/MyDrive/RL_models/sac_adr")

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
[ADR] mass=12.79 range=[6.60,13.10] type=interior
[ADR] mass=10.64 range=[6.60,13.10] type=interior
[ADR] mass=13.10 range=[6.60,13.10] type=high
[ADR] mass=13.10 range=[6.60,13.10] type=high
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 38.2     |
|    ep_rew_mean     | -3.2     |
|    success_rate    | 0.24     |
| time/              |          |
|    episodes        | 2744     |
|    fps             | 42       |
|    time_elapsed    | 2678     |
|    total_timesteps | 112820   |
| train/             |          |
|    actor_loss      | 7.9      |
|    critic_loss     | 0.0336   |
|    ent_coef        | 0.00223  |
|    ent_coef_loss   | -1.57    |
|    learning_rate   | 0.0001   |
|    n_updates       | 112719   |
---------------------------------
[ADR] mass=7.53 range=[6.60,13.10] type=interior
[ADR] mass=10.05 range=[6.60,13.10] type=interior
[ADR] mass=10.92 ra

In [ ]:
# training got interrupted so we continue from last checkpoint:
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from google.colab import drive

drive.mount('/content/drive')

# initializing the source environment wrapped with ADR
env = gym.make(
    "PandaPush-v3",
    render_mode="rgb_array",
    reward_type="dense",
    type="source"
)
env = RandomizationWrapper(env, mode="ADR")

# creating a CallBack instance to save progress
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path="/content/drive/MyDrive/RL_models/checkpoints_adr/",
)
# loading last checkpoint checkpoint
SAC_adr = SAC.load(
    "/content/drive/MyDrive/RL_models/checkpoints_adr/rl_model_500000_steps",
    env=env
)

SAC_adr.learn(
    total_timesteps=500_000,  # remaining steps to each a million
    callback=checkpoint_callback,
    reset_num_timesteps=False  # to continue the counter from where we were
)

SAC_adr.save("/content/drive/MyDrive/RL_models/sac_adr")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Mounted at /content/drive
Created object with mass: 1.0
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
[ADR] mass=0.10 range=[0.10,20.00] type=low
Logging to ./tb_logs/SAC_0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
[ADR] mass=10.10 range=[10.10,10.10] type=interior
[ADR] mass=10.10 range=[10.10,10.10] type=high
[ADR] Updated mass_max=10.10
[ADR] mass=10.10 range=[10.10,10.10] type=high
[ADR] mass=10.10 range=[10.10,10.10] type=high
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 30.4     |
|    ep_rew_mean     | -2.51    |
|    success_rate    | 0.55     |
| time/              |          |
|    episodes        | 17556    |
|    fps             | 29       |
|    time_elapsed    | 6128     |
|    total_timesteps | 678636   |
| train/             |          |
|    actor_loss      | 6.83     |
|    critic_loss     | 0.0702   |
|    ent_coef        | 0.00439  |
|    ent_coef_loss   | -0.648   |
|    learning_rate   | 0.0001   |
|    n_updates       | 678533   |
---------------------------------
[ADR] mass=10.10 range=[10.10,10.10] type=high
[ADR] mass=10.10 range=[10.10,10.10] type=

KeyboardInterrupt: 

We **interrupted** training at 700,000 steps because the ADR mass interval had **collapsed to [10.0, 10.0], meaning no randomization was occurring anymore**. This collapse was caused by a poor initialization. Since the starting mass distribution [0.1, 20] was very wide, the agent consistently performed below t_low on the lower boundary, causing mass_min to increase at each update. The same thing was happening on the mass' upper bound. This caused both masses to reach a value of 10,  making the target mass of 5 kg  excluded from the interval as well. This illustrates a key limitation of ADR: it is highly sensitive to initialization and hyperparameter choices (delta, t_low, t_high). The solution would be to follow the recommendatio  of the AutoDR paper and start at a narrow interval
### **Training an ADR agent with a SAC model in source environment with a narrow starting interval**

In [ ]:
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from google.colab import drive

drive.mount('/content/drive')

# initializing the source environment wrapped with ADR
env = gym.make(
    "PandaPush-v3",
    render_mode="rgb_array",
    reward_type="dense",
    type="source"
)
env = RandomizationWrapper(env, mode="ADR", mass_range=(0.9, 1.1), delta=0.1,
    t_low=0.4,
    t_high=0.6)



# creating a CallBack instance to save progress
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path="/content/drive/MyDrive/RL_models/checkpoints_adr_narrow/",
)

# creating an instance of the SAC model and do training on selected env
SAC_adr_narrow = SAC(
    "MultiInputPolicy",
    env,
    learning_rate=3e-4,
    gamma = 0.98,
    batch_size=256,
    verbose=1,
    tensorboard_log="./tb_logs/",
    seed = 42
)
SAC_adr_narrow.learn(total_timesteps=1_000_000, callback=checkpoint_callback)

SAC_adr_narrow.save("/content/drive/MyDrive/RL_models/sac_adr_narrow")

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 7.35     |
|    ep_rew_mean     | -0.551   |
|    success_rate    | 1        |
| time/              |          |
|    episodes        | 35700    |
|    fps             | 28       |
|    time_elapsed    | 20239    |
|    total_timesteps | 574222   |
| train/             |          |
|    actor_loss      | 1.71     |
|    critic_loss     | 0.06     |
|    ent_coef        | 0.0055   |
|    ent_coef_loss   | 0.0168   |
|    learning_rate   | 0.0003   |
|    n_updates       | 574121   |
---------------------------------
[ADR] mass=0.10 range=[0.10,30.00] type=low
[ADR] mass=30.00 range=[0.10,30.00] type=high
[ADR] mass=30.00 range=[0.10,30.00] type=high
[ADR] mass=22.19 range=[0.10,30.00] type=interior
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 7.59     |
|    ep_rew_mean     |

In [ ]:
# training got interrupted so we continue from last checkpoint:
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from google.colab import drive

drive.mount('/content/drive')

# initializing the source environment wrapped with ADR
env = gym.make(
    "PandaPush-v3",
    render_mode="rgb_array",
    reward_type="dense",
    type="source"
)
env = RandomizationWrapper(env, mode="ADR", mass_range=(0.1, 4.90), delta=0.1,
    t_low=0.4,
    t_high=0.6)

# creating a CallBack instance to save progress
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path="/content/drive/MyDrive/RL_models/checkpoints_adr_narrow/",
)
# loading last checkpoint checkpoint
SAC_adr_narrow = SAC.load(
    "/content/drive/MyDrive/RL_models/checkpoints_adr_narrow/rl_model_380000_steps",
    env=env,
    seed = 42
)

SAC_adr_narrow.learn(
    total_timesteps=620_000,  # remaining steps to each a million
    callback=checkpoint_callback,
    reset_num_timesteps=False  # to continue the counter from where we were
)

SAC_adr_narrow.save("/content/drive/MyDrive/RL_models/sac_adr_narrow")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Created object with mass: 1.0
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[ADR] mass=4.90 range=[0.10,4.90] type=high
Logging to ./tb_logs/SAC_0
[ADR] mass=1.59 range=[0.10,4.90] type=interior
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 39.6     |
|    ep_rew_mean     | -3.57    |
|    success_rate    | 0.22     |
| time/              |          |
|    episodes        | 9436     |
|    fps             | 12       |
|    time_elapsed    | 4        |
|    total_timesteps | 380050   |
| train/             |          |
|    actor_loss      | 5.21     |
|    critic_loss     | 0.0133   |
|    ent_coef        | 0.00205  |
|    ent_coef_loss   | -1.32    |
|    learning_rate   | 0.0001   |
|    n_updates       | 379948   |
---------------------------------
[ADR] mass=4.90 range=[0.10,4.90] type=high
[ADR] mass=4.54 range=[0.10,4.90] type=interior
[ADR] mass=0.10 range=[0.10,4.90] type=low
[ADR] mass=4.31 range=[0.10,4.90] type=interior
---------------------------------
| rollout/           |          |
|    ep_len_mean  

KeyboardInterrupt: 

### **Evaluating performances of both models in source and target environments**

Just like for previous tasks, we will be using the following function

In [ ]:
import numpy as np
import pandas as pd
from stable_baselines3 import SAC
import panda_gym


def evaluate(model, n_episodes=50, deterministic=True, env_type="source"):
    env = gym.make("PandaPush-v3", render_mode="rgb_array", type=env_type, reward_type="dense")

    episode_returns = []
    successes = []
    for episode in range(1, n_episodes + 1):
        obs, info = env.reset()
        terminated = False
        truncated = False
        episode_return = 0.0
        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = env.step(action)
            episode_return += float(reward)
        episode_returns.append(episode_return)
        if isinstance(info, dict) and "is_success" in info:
            successes.append(float(info["is_success"]))

    env.close()

    returns = np.array(episode_returns, dtype=np.float32)
    success_rate = float(np.mean(successes)) if successes else 0.0

    return returns.mean(), returns.std(), success_rate

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# loading the trained models
from google.colab import drive

drive.mount('/content/drive')
SAC_udr = SAC.load("/content/drive/MyDrive/RL_models/sac_udr_narrow")
SAC_adr = SAC.load("/content/drive/MyDrive/RL_models/sac_adr_narrow",)

# évaluating the 4 configurations
configs = [
    ("UDR", SAC_udr, "source"),
    ("UDR", SAC_udr, "target"),
    ("ADR", SAC_adr, "source"),
    ("ADR", SAC_adr, "target"),
]

results = []
for method, model, env_type in configs:
    print(f"Evaluating {method} on {env_type}...")
    mean, std, sr = evaluate(model, n_episodes=50, env_type=env_type)
    results.append({
        "Method": method,
        "Train→Test": f"source:{env_type}",
        "Mean Return": f"{mean:.3f}",
        "Std Return": f"{std:.3f}",
        "Success Rate": f"{sr:.2%}",
    })

df = pd.DataFrame(results)
print("\n=== Domain Randomization Results ===")
print(df.to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Evaluating UDR on source...
Created object with mass: 1.0
Evaluating UDR on target...
Created object with mass: 5.0
Evaluating ADR on source...
Created object with mass: 1.0
Evaluating ADR on target...
Created object with mass: 5.0

=== Domain Randomization Results ===
Method    Train→Test Mean Return Std Return Success Rate
   UDR source:source      -2.108      2.184       64.00%
   UDR source:target      -1.903      1.568       64.00%
   ADR source:source      -0.561      0.685       98.00%
   ADR source:target      -0.507      0.386      100.00%


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### **testing sensitivity**
since results seemed skewed, we opted for a different test where the mass is shifted from one test episode to another using a variant of the testing function at hand

In [ ]:
def evaluate(
    model,
    n_episodes=50,
    deterministic=True,
    env_type="source",
    mass_low=0.1,
    mass_high=30.0,
):
    env = gym.make("PandaPush-v3", render_mode="rgb_array", type=env_type, reward_type="dense")

    sim = env.unwrapped.task.sim
    physics_client = sim.physics_client

    episode_returns = []
    successes = []

    for episode in range(1, n_episodes + 1):
        obs, info = env.reset()
        object_body_id = sim._bodies_idx["object"]

        # Resampling the mass
        sampled_mass = np.random.uniform(mass_low, mass_high)
        physics_client.changeDynamics(
            bodyUniqueId=object_body_id,
            linkIndex=-1,
            mass=float(sampled_mass),
        )

        # we print the mass we test at every episode
        actual_mass = physics_client.getDynamicsInfo(object_body_id, -1)[0]
        print(f"  Episode {episode} — masse appliquée : {actual_mass:.3f} kg")

        terminated = False
        truncated = False
        episode_return = 0.0
        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = env.step(action)
            episode_return += float(reward)

        episode_returns.append(episode_return)
        if isinstance(info, dict) and "is_success" in info:
            successes.append(float(info["is_success"]))

    env.close()

    returns = np.array(episode_returns, dtype=np.float32)
    success_rate = float(np.mean(successes)) if successes else 0.0

    return returns.mean(), returns.std(), success_rate

In [ ]:
# loading the trained models
from google.colab import drive

drive.mount('/content/drive')
# Chargement des modèles
drive.mount('/content/drive')
SAC_udr = SAC.load("/content/drive/MyDrive/RL_models/sac_udr_narrow")
SAC_adr = SAC.load("/content/drive/MyDrive/RL_models/checkpoints_adr_narrow/rl_model_580000_steps")

# Évaluation des 4 configurations
configs = [
    ("UDR", SAC_udr, "source"),
    ("UDR", SAC_udr, "target"),
    ("ADR", SAC_adr, "source"),
    ("ADR", SAC_adr, "target"),
]

results = []
for method, model, env_type in configs:
    print(f"Evaluating {method} on {env_type}...")
    mean, std, sr = evaluate(model, n_episodes=50, env_type=env_type)
    results.append({
        "Method": method,
        "Train→Test": f"source→{env_type}",
        "Mean Return": f"{mean:.3f}",
        "Std Return": f"{std:.3f}",
        "Success Rate": f"{sr:.2%}",
    })

df = pd.DataFrame(results)
print("\n=== Domain Randomization Results ===")
print(df.to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Evaluating UDR on source...
Created object with mass: 1.0
  Episode 1 — masse appliquée : 11.299 kg
  Episode 2 — masse appliquée : 28.526 kg
  Episode 3 — masse appliquée : 21.987 kg
  Episode 4 — masse appliquée : 18.000 kg
  Episode 5 — masse appliquée : 4.765 kg
  Episode 6 — masse appliquée : 4.764 kg
  Episode 7 — masse appliquée : 1.837 kg
  Episode 8 — masse appliquée : 25.999 kg
  Episode 9 — masse appliquée : 18.073 kg
  Episode 10 — masse appliquée : 21.271 kg
  Episode 11 — masse appliquée : 0.715 kg
  Episode 12 — masse appliquée : 29.100 kg
  Episode 13 — masse appliquée : 24.990 kg
  Episode 14 — masse appliquée : 6.449 kg
  Episode 15 — masse appliquée : 5.537 kg
  Episode 16 — masse appliquée : 5.584 kg
  Episode 17

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
